In [1]:
!pip install docling==2.58.0 pypdfium2==4.30.0 pymilvus llama_stack_client==0.4.2 requests tqdm

In [2]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from llama_stack_client import LlamaStackClient
import os
from pymilvus import MilvusClient
from tqdm import tqdm
import time

In [3]:
source = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"

# Create PDF pipeline options
pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False  # 👈 Disable OCR

# Configure converter with format options
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
)

start = time.time()

doc = converter.convert(source).document

markdown_content = doc.export_to_markdown()
output_path = "Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.md"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Saved to {output_path}")

INFO:docling.datamodel.document:detected formats: [<InputFormat.CSV: 'csv'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['vlm', 'api']
INFO:docling.pipeline.base_pipeline:Processing document Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv
INFO:docling.backend.csv_backend:Parsing CSV with delimiter: ","
INFO:docling.backend.csv_backend:Detected 681 lines
INFO:docling.document_converter:Finished converting document Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv in 0.08 sec.


Saved to Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.md


In [4]:
def add_to_collection(collection, chunks, embeddings, batch_size=5000):
    for i in range(0, len(chunks), batch_size):
        end = min(i + batch_size, len(chunks))
        collection.add(
            documents=chunks[i:end],
            embeddings=embeddings[i:end].tolist() if hasattr(embeddings, 'tolist') else embeddings[i:end],
            ids=[f"chunk_{j}" for j in range(i, end)]
        )
    print(f"✅ Loaded {collection.count()} chunks into the vector store")

In [5]:
with open(output_path, "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Document length: {len(full_text):,} characters")
print(f"\nFirst 500 characters:\n{full_text[:500]}...")

Document length: 357,367 characters

First 500 characters:
|   YEAR | QUARTER   | SKU       | SKU_Description                                                                                                                                                                                                              | Product                                  | Currency   | List_Price   | Unit_of_Measure               | Cores     | Nodes     |   Sockets | Virtual_Guests   | Support_Level   | Support_Type   | Category                       | Region   | Count...


In [6]:
def chunk_text_markdown(text, chunk_size=1000, overlap=200):
    """Split markdown into chunks that respect structural boundaries.
    
    Splits on headings and paragraph breaks first, then accumulates
    blocks into chunks within the size budget. Falls back to word-boundary
    splitting only when a single block exceeds chunk_size.
    """
    import re
    
    # Split on markdown headings (keep the heading with the text that follows)
    # and on double newlines (paragraph boundaries)
    sections = re.split(r'(?=\n#{1,6} )', text)
    
    # Further split each section on paragraph breaks (double newline)
    blocks = []
    for section in sections:
        paragraphs = re.split(r'\n{2,}', section)
        for p in paragraphs:
            stripped = p.strip()
            if stripped:
                blocks.append(stripped)
    
    # Accumulate blocks into chunks
    chunks = []
    current_chunk = ""
    
    for block in blocks:
        # If a single block exceeds chunk_size, split it at word boundaries
        if len(block) > chunk_size:
            # Flush current chunk first
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
                current_chunk = ""
            
            # Word-boundary fallback for oversized blocks
            start = 0
            while start < len(block):
                end = start + chunk_size
                if end < len(block):
                    space_pos = block.rfind(" ", start, end)
                    if space_pos > start:
                        end = space_pos
                sub = block[start:end].strip()
                if sub:
                    chunks.append(sub)
                next_start = end - overlap
                start = max(next_start, start + 1)
            continue
        
        # Would adding this block exceed the budget?
        candidate = (current_chunk + "\n\n" + block).strip() if current_chunk else block
        if len(candidate) > chunk_size and current_chunk.strip():
            chunks.append(current_chunk.strip())
            current_chunk = block
        else:
            current_chunk = candidate
    
    # Flush the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    # Apply overlap: prepend tail of previous chunk to each subsequent chunk
    if overlap > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = chunks[i - 1]
            # Take the last `overlap` characters, back up to word boundary
            if len(prev) > overlap:
                tail_start = len(prev) - overlap
                space_pos = prev.find(" ", tail_start)
                if space_pos != -1:
                    tail = prev[space_pos:].strip()
                else:
                    tail = prev[tail_start:].strip()
            else:
                tail = prev.strip()
            overlapped.append((tail + "\n\n" + chunks[i]).strip())
        chunks = overlapped
    
    return chunks

chunks = chunk_text_markdown(full_text, chunk_size=1000, overlap=200)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")

Total chunks created: 449
Average chunk length: 1131 characters


In [7]:
for size in [200, 500, 1000, 2000]:
    test_chunks = chunk_text_markdown(full_text, chunk_size=size, overlap=int(size * 0.2))
    print(f"Chunk size: {size:>5}  |  Overlap: {int(size*0.2):>4}  |  Total chunks: {len(test_chunks):>4}")

Chunk size:   200  |  Overlap:   40  |  Total chunks: 2308
Chunk size:   500  |  Overlap:  100  |  Total chunks:  998
Chunk size:  1000  |  Overlap:  200  |  Total chunks:  449
Chunk size:  2000  |  Overlap:  400  |  Total chunks:  224


In [8]:
# Initialize the Llama Stack client
client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_SERVER_URL", "http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321")
)

milvus_client = MilvusClient(uri="http://milvus-service.proposal-rh-ai.svc.cluster.local:19530")

vector_db_skus_name = "skus_rh_vector_db"

In [9]:
vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

for vector_store in vector_stores:
    client.vector_stores.delete(
        vector_store_id=vector_store.id
    )
    print(f"Vector store: {vector_store.name} deleted")

print("All Vector stores deleted")

vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id=None, object='list', first_id=None)
All Vector stores deleted
Vector stores SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id=None, object='list', first_id=None)


In [ ]:
#if milvus_client.has_collection(collection_name):
#    milvus_client.drop_collection(collection_name)

In [10]:
# Create a vector store skus and index the file
vector_store_skus = client.vector_stores.create(
    name=vector_db_skus_name,
    extra_body={
        "provider_id": "milvus-remote",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


In [23]:
vector_db_skus_id = ""
collection_name = ""
vector_dbs = client.vector_stores.list()
for vector_db in vector_dbs:
    if vector_db.name == vector_db_skus_name:
        vector_db_skus_id = vector_db.id
        collection_name = vector_db_skus_id.replace("-", "_")
        break
if vector_db_skus_id == "":
    print(f"Vector DB ID for SKUs: {vector_db_skus_name} not found in the vector stores")

print(f"Vector DB ID for SKUs: {vector_db_skus_id}")
print(f"Collection Name of Milvus for SKUs: {collection_name}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector DB ID for SKUs: vs_4e800ce5-334c-4377-8d3f-bbbe6e7f39e8
Collection Name of Milvus for SKUs: vs_4e800ce5_334c_4377_8d3f_bbbe6e7f39e8


In [24]:
collections_milvus = milvus_client.list_collections()

for collection in collections_milvus:
    print(f"collection_name: {collection}")

if len(collections_milvus) == 0:
    print("collections empty")

collection_name: vs_4e800ce5_334c_4377_8d3f_bbbe6e7f39e8


In [14]:
def emb_text(text):
    return (
        client.embeddings.create(input=text, model="sentence-transformers/ibm-granite/granite-embedding-125m-english")
        .data[0]
        .embedding
    )

In [15]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=768,
    metric_type="IP",  # Inner product distance
    consistency_level="Bounded",  # Supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/tune_consistency.md#Consistency-Level for more details.
)

In [16]:
data = []

for i, line in enumerate(tqdm(chunks, desc="Creating embeddings")):
    data.append({"id": i, "vector": emb_text(line), "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 449/449 [01:26<00:00,  5.22it/s]


{'insert_count': 449, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 

In [30]:
question = "Can you search and list the SKU_Description of product Advanced Cluster Management"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=[
        emb_text(question)
    ],  # Use the `emb_text` function to convert the question to an embedding vector
    limit=3,  # Return top 3 results
    search_params={"metric_type": "IP", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
)

import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/embeddings "HTTP/1.1 200 OK"


[
    [
        "| ACM - Advanced Cluster Management        | USD        | 9.900,00     | PHYSICAL NODE                 | 0         | 0         |         2 | 0                | L1-L3           | Premium        |\n\n| ACM - Advanced Cluster Management        | USD        | 9.900,00     | PHYSICAL NODE                 | 0         | 0         |         2 | 0                | L1-L3           | Premium        | SUBSCRIPTIONS                  | LATAM    | ALL       | 1 YEARS        |\n|   2025 | Q3        | MW04497   | Red Hat Advanced Cluster Management for Kubernetes (Bare Metal Node), Standard (1-2 Sockets up to 128 Cores)                                                                                                                 | ACM - Advanced Cluster Management        | USD        | 6.600,00     | PHYSICAL NODE                 | 0         | 0         |         2 | 0                | L1-L3           | Standard       | SUBSCRIPTIONS                  | LATAM    | ALL       | 1 YEARS  

In [34]:
query = "List of Red Hat OpenShift Container Platform SKU"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_skus_id],
    }],
)

print(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"


RAG Query from skus_rh_vector_db - Result: 
The Red Hat OpenShift Container Platform SKU information is not available in the provided database. However, I can suggest some possible SKUs based on the official Red Hat website:

* OpenShift Container Platform 4.11: This SKU includes access to the latest version of OpenShift, as well as support and maintenance for 1 year.
* OpenShift Container Platform 4.10: This SKU includes access to the previous version of OpenShift, as well as support and maintenance for 1 year.
* OpenShift Container Platform 4.9: This SKU includes access to an earlier version of OpenShift, as well as support and maintenance for 1 year.

Please note that these SKUs are subject to change and may not be up-to-date. For the most accurate and current information, please visit the official Red Hat website or contact their sales team directly.


In [35]:
query = "Can you search and list the SKU, SKU_Description, List_Price, Currenc of product Advanced Cluster Management"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_skus_id],
    }],
)

print(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"


RAG Query from skus_rh_vector_db - Result: 
I'm sorry, but I could not find any information on the SKU, SKU_Description, List_Price, Currency of product Advanced Cluster Management. Can I help you with anything else?
